# Predicting NFL Game Outcomes Using Data Science
**Independent Sports Analytics Project | 2015–2025**

**Research question:** Which NFL statistics are associated with winning, and can recent team performance help predict future outcomes?

**Hypothesis:** Turnover differential would be more strongly associated with winning than passing- or rushing-yard differential.

**Approach:** Compare same-game outcomes descriptively, then predict later games using only five-game pregame averages. Train on 2015–2023 and test on 2024–2025.

**Tools:** Python, pandas, NumPy, Matplotlib, nflreadpy, scikit-learn.

## 1. Import Libraries & Load Data
Download NFL team statistics and game results. The team-statistics dataset is loaded across seasons, then filtered below.

In [ ]:
!pip -q install nflreadpy

### Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nflreadpy as nfl
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
url = "https://raw.githubusercontent.com/nflverse/nfldata/master/data/games.csv"

games = pd.read_csv(url)

print("Rows:", len(games))
print("Columns:", len(games.columns))

games.head()

In [ ]:
# Load all available NFL team statistics
team_stats = nfl.load_team_stats(seasons=True).to_pandas()

print("Rows:", len(team_stats))
print("Columns:", len(team_stats.columns))

print("\nSeasons:")
print(sorted(team_stats["season"].unique()))

## 2. Data Cleaning & Preparation
Match team statistics to game results, keep 2015–2025 games with a final score, exclude ties, and construct opponent-matched differentials. **Turnover differential uses turnovers actually lost**, not merely forced fumbles.

In [ ]:
# Keep only the game information we need
game_results = games[
    ["game_id", "away_team", "home_team", "away_score", "home_score"]
].copy()

# Merge game results with our team statistics
team_data = team_stats.merge(
    game_results,
    on="game_id",
    how="left"
)

print("Rows:", len(team_data))
team_data.head()

In [ ]:
# Determine each team's score and their opponent's score

team_data["team_score"] = np.where(
    team_data["team"] == team_data["home_team"],
    team_data["home_score"],
    team_data["away_score"]
)

team_data["opponent_score"] = np.where(
    team_data["team"] == team_data["home_team"],
    team_data["away_score"],
    team_data["home_score"]
)

team_data[[
    "game_id",
    "team",
    "opponent_team",
    "team_score",
    "opponent_score"
]].head(10)

In [ ]:
team_data = team_data[team_data["season"].between(2015, 2025)].copy()
# Remove games without recorded final scores and tied games.
team_data = team_data.dropna(subset=["team_score", "opponent_score"]).copy()
team_data = team_data[team_data["team_score"] != team_data["opponent_score"]].copy()
team_data["win"] = (team_data["team_score"] > team_data["opponent_score"]).astype(int)
print("Seasons:", sorted(team_data["season"].unique()))
print("Team-game observations:", len(team_data))
print(team_data["win"].value_counts())

In [ ]:
# Create a smaller dataset containing each team's passing and rushing stats
opponent_stats = team_data[[
    "game_id",
    "team",
    "passing_yards",
    "rushing_yards"
]].copy()

# Rename the team/stat columns so we know these belong to the opponent
opponent_stats = opponent_stats.rename(columns={
    "team": "opponent_team",
    "passing_yards": "opponent_passing_yards",
    "rushing_yards": "opponent_rushing_yards"
})

# Match each team with its opponent using game_id + opponent_team
team_data = team_data.merge(
    opponent_stats,
    on=["game_id", "opponent_team"],
    how="left"
)

print(team_data[[
    "game_id",
    "team",
    "opponent_team",
    "passing_yards",
    "opponent_passing_yards",
    "rushing_yards",
    "opponent_rushing_yards"
]].head(10))

In [ ]:
# Calculate the difference between each team's production
# and its opponent's production

team_data["passing_yard_diff"] = (
    team_data["passing_yards"] -
    team_data["opponent_passing_yards"]
)

team_data["rushing_yard_diff"] = (
    team_data["rushing_yards"] -
    team_data["opponent_rushing_yards"]
)

# Check our new variables
team_data[[
    "team",
    "opponent_team",
    "passing_yard_diff",
    "rushing_yard_diff",
    "win"
]].head(10)

In [ ]:
team_data["turnovers_committed"] = team_data["passing_interceptions"] + team_data["fumbles_lost_total"]

In [ ]:
opponent_turnovers = team_data[[
    "game_id",
    "team",
    "passing_interceptions",
    "fumbles_lost_total"
]].copy()

opponent_turnovers["opponent_turnovers_committed"] = (
    opponent_turnovers["passing_interceptions"] +
    opponent_turnovers["fumbles_lost_total"]
)

opponent_turnovers = opponent_turnovers[[
    "game_id",
    "team",
    "opponent_turnovers_committed"
]]

opponent_turnovers = opponent_turnovers.rename(
    columns={"team": "opponent_team"}
)

team_data = team_data.merge(
    opponent_turnovers,
    on=["game_id", "opponent_team"],
    how="left"
)

team_data["turnover_diff"] = (
    team_data["opponent_turnovers_committed"] -
    team_data["turnovers_committed"]
)

## 3. Exploratory Data Analysis
These comparisons describe **completed games**; they do not establish causation or constitute pregame predictions.

In [ ]:
# Group teams by turnover differential
turnover_analysis = (
    team_data
    .groupby("turnover_diff")["win"]
    .agg(["count", "mean"])
    .reset_index()
)

# Convert win rate from decimal to percentage
turnover_analysis["win_rate"] = turnover_analysis["mean"] * 100

turnover_analysis

In [ ]:
print(
    "Win rate when passing-yard differential is positive:",
    team_data.loc[team_data["passing_yard_diff"] > 0, "win"].mean() * 100
)

print(
    "Win rate when passing-yard differential is negative:",
    team_data.loc[team_data["passing_yard_diff"] < 0, "win"].mean() * 100
)

In [ ]:
print(
    "Win rate when rushing-yard differential is positive:",
    team_data.loc[team_data["rushing_yard_diff"] > 0, "win"].mean() * 100
)

print(
    "Win rate when rushing-yard differential is negative:",
    team_data.loc[team_data["rushing_yard_diff"] < 0, "win"].mean() * 100
)

In [ ]:
print(
    "Positive turnover differential:",
    team_data.loc[team_data["turnover_diff"] > 0, "win"].mean() * 100
)

print(
    "Negative turnover differential:",
    team_data.loc[team_data["turnover_diff"] < 0, "win"].mean() * 100
)

print(
    "Zero turnover differential:",
    team_data.loc[team_data["turnover_diff"] == 0, "win"].mean() * 100
)

**Interpretation:** Same-game turnover differential is strongly associated with winning. Rushing-yard differential also has a strong association, but game situation may affect rushing volume; teams ahead late in games may run more.

## 4. Data Visualizations
Compare win rates when a team finishes with a positive versus negative statistical differential.

In [ ]:
comparison_data = pd.DataFrame({
    "Factor": [
        "Turnover differential",
        "Rushing-yard differential",
        "Passing-yard differential"
    ],
    "Positive": [
        team_data.loc[team_data["turnover_diff"] > 0, "win"].mean() * 100,
        team_data.loc[team_data["rushing_yard_diff"] > 0, "win"].mean() * 100,
        team_data.loc[team_data["passing_yard_diff"] > 0, "win"].mean() * 100
    ],
    "Negative": [
        team_data.loc[team_data["turnover_diff"] < 0, "win"].mean() * 100,
        team_data.loc[team_data["rushing_yard_diff"] < 0, "win"].mean() * 100,
        team_data.loc[team_data["passing_yard_diff"] < 0, "win"].mean() * 100
    ]
})

comparison_data

In [ ]:
ax = comparison_data.plot(x="Factor", y=["Positive", "Negative"], kind="bar", figsize=(10, 6))
ax.set_ylabel("Win Rate (%)")
ax.set_xlabel("")
ax.set_title("NFL Win Rate: Positive vs. Negative Statistical Differentials")
ax.set_ylim(0, 100)
ax.tick_params(axis="x", rotation=0)
ax.axhline(50, linestyle="--", color="gray", label="_nolegend_")
ax.legend(title="Performance vs. Opponent")
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)
plt.tight_layout()
plt.savefig("nfl_win_rate_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Pregame Feature Engineering
For each team, calculate its average statistical differential over up to five **previous** games in the same season. The `.shift(1)` operation ensures the current game is excluded. Then subtract the opponent’s corresponding pregame average.

In [ ]:
team_data = team_data.sort_values(
    ["team", "season", "week"]
).copy()

In [ ]:
team_data["pregame_turnover_diff"] = (
    team_data.groupby(["team", "season"])["turnover_diff"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

team_data["pregame_rushing_diff"] = (
    team_data.groupby(["team", "season"])["rushing_yard_diff"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

team_data["pregame_passing_diff"] = (
    team_data.groupby(["team", "season"])["passing_yard_diff"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean())
)

In [ ]:
opponent_pregame = team_data[[
    "game_id",
    "team",
    "pregame_turnover_diff",
    "pregame_rushing_diff",
    "pregame_passing_diff"
]].copy()

opponent_pregame = opponent_pregame.rename(columns={
    "team": "opponent_team",
    "pregame_turnover_diff": "opponent_pregame_turnover_diff",
    "pregame_rushing_diff": "opponent_pregame_rushing_diff",
    "pregame_passing_diff": "opponent_pregame_passing_diff"
})

In [ ]:
team_data = team_data.merge(
    opponent_pregame,
    on=["game_id", "opponent_team"],
    how="left"
)

In [ ]:
team_data["pregame_turnover_advantage"] = (
    team_data["pregame_turnover_diff"] -
    team_data["opponent_pregame_turnover_diff"]
)

team_data["pregame_rushing_advantage"] = (
    team_data["pregame_rushing_diff"] -
    team_data["opponent_pregame_rushing_diff"]
)

team_data["pregame_passing_advantage"] = (
    team_data["pregame_passing_diff"] -
    team_data["opponent_pregame_passing_diff"]
)

In [ ]:
model_data = team_data.dropna(subset=[
    "pregame_turnover_advantage",
    "pregame_rushing_advantage",
    "pregame_passing_advantage"
]).copy()

print("Total rows before removing missing values:", len(team_data))
print("Rows available for prediction model:", len(model_data))
print("Wins:", model_data["win"].sum())
print("Losses:", len(model_data) - model_data["win"].sum())

**Missing values:** First games of a season have no prior in-season statistics and are excluded from the predictive dataset.

## 6. Machine Learning Model
Use a chronological split: 2015–2023 for training and 2024–2025 for testing. Fit scaling parameters on training data only.

In [ ]:
train_data = model_data[
    model_data["season"] <= 2023
].copy()

test_data = model_data[
    model_data["season"].isin([2024, 2025])
].copy()

In [ ]:
print("Training rows:", len(train_data))
print("Testing rows:", len(test_data))

print("\nTraining seasons:")
print(sorted(train_data["season"].unique()))

print("\nTesting seasons:")
print(sorted(test_data["season"].unique()))

In [ ]:
features = [
    "pregame_turnover_advantage",
    "pregame_rushing_advantage",
    "pregame_passing_advantage"
]

X_train = train_data[features]
y_train = train_data["win"]

X_test = test_data[features]
y_test = test_data["win"]

In [ ]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

In [ ]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Model Accuracy:", accuracy)
print("Model Accuracy (%):", accuracy * 100)

### Standardized logistic regression
Standardizing allows more meaningful comparisons of coefficients measured in different units.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [ ]:
scaled_model = LogisticRegression(max_iter=1000)

scaled_model.fit(X_train_scaled, y_train)

scaled_predictions = scaled_model.predict(X_test_scaled)

from sklearn.metrics import accuracy_score

scaled_accuracy = accuracy_score(y_test, scaled_predictions)

print("Standardized Model Accuracy:", scaled_accuracy * 100)

In [ ]:
standardized_coefficients = pd.DataFrame({
    "Feature": features,
    "Standardized Coefficient": scaled_model.coef_[0]
})

standardized_coefficients["Absolute Coefficient"] = (
    standardized_coefficients["Standardized Coefficient"].abs()
)

standardized_coefficients = standardized_coefficients.sort_values(
    "Absolute Coefficient",
    ascending=False
)

standardized_coefficients

## 7. Model Evaluation & Validation
Compare against a majority-class baseline, inspect the confusion matrix, evaluate by season, and verify that each unique game is represented twice with consistent opposing predictions.

In [ ]:
baseline_accuracy = y_test.value_counts(normalize=True).max()

print(f"Baseline Accuracy: {baseline_accuracy * 100:.2f}%")
print(f"Model Accuracy: {scaled_accuracy * 100:.2f}%")

print(
    f"Improvement: {(scaled_accuracy - baseline_accuracy) * 100:.2f} percentage points"
)

In [ ]:
cm = confusion_matrix(y_test, predictions)

print("Confusion Matrix:")
print(cm)

In [ ]:
test_results = test_data.copy()

test_results["prediction"] = scaled_predictions

season_accuracy = (
    test_results.groupby("season")
    .apply(
        lambda x: (x["prediction"] == x["win"]).mean() * 100,
        include_groups=False
    )
    .reset_index(name="Accuracy (%)")
)

print(season_accuracy)

In [ ]:
print("Team-game observations:", len(test_data))
print("Unique NFL games:", test_data["game_id"].nunique())

print("\nObservations per game:")
print(test_data.groupby("game_id").size().value_counts())

In [ ]:
game_predictions = test_data[[
    "game_id", "team", "win"
]].copy()

game_predictions["predicted_win"] = scaled_predictions

# Verify exactly one team is predicted to win each game
prediction_check = game_predictions.groupby("game_id").agg(
    actual_wins=("win", "sum"),
    predicted_wins=("predicted_win", "sum")
)

print("Games with inconsistent predictions:",
      (prediction_check["predicted_wins"] != 1).sum())

# Keep one team perspective per game
unique_games = game_predictions.drop_duplicates("game_id")

game_accuracy = (
    unique_games["win"] == unique_games["predicted_win"]
).mean()

print(f"Unique-game accuracy: {game_accuracy * 100:.2f}%")

## 8. Conclusions & Limitations

**Findings:** In the original analysis, positive same-game turnover differential had the largest observed win-rate gap of the three statistics. In the pregame logistic regression, standardized passing and rushing advantages had slightly larger coefficients than turnover advantage. These are different questions: explaining completed games versus forecasting later games.

**Model evaluation (previous run):** 63.50% accuracy on 537 unique games from 2024–2025, compared with a 50.00% majority-class baseline. Seasonal team-game accuracy was 64.31% in 2024 and 62.69% in 2025. These results should be confirmed by rerunning this cleaned notebook.

**Limitations:** Observational associations do not prove causation. This simple model omits injuries, quarterbacks, home field, and strength of schedule. Five-game averages reset each season, excluding each team’s opening game. The dataset contains two team perspectives per game; report accuracy per **unique game** rather than treating both perspectives as independent.

**Next steps:** Add home-field advantage, try alternative pregame windows, compare against other baselines, and evaluate predicted probabilities as well as classifications.

**Data sources:** NFL team statistics via `nflreadpy` and game results from the nflverse/nfldata `games.csv` file.